<a href="https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/06_autogen_0_4_vs_crewai.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" style="height: 28px; vertical-align: middle;"/></a>  **[▶️ Run this notebook in Colab](https://colab.research.google.com/github/hassan11196/llm-systems-cookbook/blob/main/notebooks/04_agents/06_autogen_0_4_vs_crewai.ipynb)**

# AutoGen vs CrewAI — two multi-agent idioms

> **Track 04 — Agents · Notebook 06 · Runtime: ≈30 s on CPU**
>
> **Prerequisites:** `04_agents/03` (LangGraph state machines).
>
> **References:**
> - [AutoGen 0.4 docs](https://microsoft.github.io/autogen/stable/)
>   (conversation-driven multi-agent).
> - [CrewAI docs](https://docs.crewai.com/) (role-based multi-agent).

---

## What

Both frameworks orchestrate multiple LLM-backed agents toward a goal;
they differ in the *primitive*:

- **AutoGen** — agents communicate by sending messages. A group chat
  manager dispatches. Good for collaborative back-and-forth.
- **CrewAI** — agents have named *roles* (planner, researcher,
  writer); each role is assigned a *task*. Execution follows a
  task DAG. Good for structured pipelines with clear hand-offs.

Both work; the choice is stylistic. We implement minimal clones of
each and run them on the same toy task (draft + critique + revise a
short article). With the same underlying LM stubs, they reach the
same final answer — the frameworks just route the messages
differently.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

from llm_systems_cookbook._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_06_autogen_0_4_vs_crewai")


## Shared task and LM stubs

Three deterministic LM-like functions:

- `drafter(topic)` — returns a short paragraph.
- `critic(text)` — returns up to three issues.
- `reviser(text, issues)` — edits the text addressing the issues.


In [ ]:
TOPIC = "the importance of the mitochondrion in eukaryotic cells"


def drafter(topic: str) -> str:
    return ("Mitochondria are organelles in eukaryotic cells. They generate energy. "
            "They contain their own DNA. They evolved from bacteria. The End.")


def critic(text: str) -> list[str]:
    issues: list[str] = []
    if "The End" in text:
        issues.append("remove the closing 'The End' phrase")
    if "energy" in text and "ATP" not in text:
        issues.append("mention ATP when describing energy production")
    if "bacteria" in text and "endosymbiotic" not in text.lower():
        issues.append("reference the endosymbiotic theory")
    return issues


def reviser(text: str, issues: list[str]) -> str:
    revised = text.replace(" The End.", "")
    if any("ATP" in i for i in issues):
        revised = revised.replace("generate energy.", "generate ATP via oxidative phosphorylation.")
    if any("endosymbiotic" in i for i in issues):
        revised = revised.replace("evolved from bacteria.",
                                   "evolved from bacteria per the endosymbiotic theory.")
    return revised


## AutoGen-style: conversation driven

All agents write to a shared message log. The "group chat manager"
decides who speaks next based on the last message. Termination is
explicit — when the critic returns zero issues.


In [ ]:
@dataclass
class Message:
    sender: str
    content: str


def run_autogen_style(topic: str, max_turns: int = 6) -> tuple[str, list[Message]]:
    log: list[Message] = []
    current_draft = drafter(topic)
    log.append(Message("drafter", current_draft))

    for _ in range(max_turns):
        issues = critic(current_draft)
        log.append(Message("critic", "issues: " + (", ".join(issues) or "none")))
        if not issues:
            break
        current_draft = reviser(current_draft, issues)
        log.append(Message("reviser", current_draft))

    return current_draft, log


autogen_final, autogen_log = run_autogen_style(TOPIC)
print(f"AutoGen-style: {len(autogen_log)} messages")
for m in autogen_log:
    print(f"  [{m.sender}]  {m.content[:80]}")
print(f"\nfinal: {autogen_final}")


## CrewAI-style: roles + task DAG

Each agent has a role and a task list; the crew executes tasks in
order. The `Crew.run` method is a plain loop. Same functions as
AutoGen, just routed by pre-declared role.


In [ ]:
@dataclass
class Agent:
    role: str
    goal: str


@dataclass
class Task:
    description: str
    agent: Agent
    expects_input_from: list[str] = field(default_factory=list)


@dataclass
class Crew:
    agents: list[Agent]
    tasks: list[Task]

    def run(self, topic: str, max_revisions: int = 3) -> tuple[str, list[str]]:
        artefacts: dict[str, str] = {}
        traces: list[str] = []

        draft = drafter(topic)
        artefacts["draft"] = draft
        traces.append(f"drafter -> draft")

        for _ in range(max_revisions):
            issues = critic(artefacts["draft"])
            traces.append(f"critic -> issues[{len(issues)}]")
            if not issues:
                break
            artefacts["draft"] = reviser(artefacts["draft"], issues)
            traces.append(f"reviser -> draft")
        return artefacts["draft"], traces


crew = Crew(
    agents=[
        Agent("drafter", "write an initial paragraph"),
        Agent("critic", "identify issues in the paragraph"),
        Agent("reviser", "fix issues in the paragraph"),
    ],
    tasks=[
        Task("write a paragraph about the topic", Agent("drafter", "write")),
        Task("critique the paragraph", Agent("critic", "review"), expects_input_from=["drafter"]),
        Task("rewrite incorporating critiques", Agent("reviser", "fix"),
             expects_input_from=["drafter", "critic"]),
    ],
)
crewai_final, crewai_trace = crew.run(TOPIC)
print(f"CrewAI-style trace: {' -> '.join(crewai_trace)}")
print(f"\nfinal: {crewai_final}")


In [ ]:
s.check(
    "autogen_final_removes_the_end",
    lambda: "The End" not in autogen_final,
    msg=f"final = {autogen_final!r}",
)
s.check(
    "autogen_final_mentions_atp",
    lambda: "ATP" in autogen_final,
    msg=f"final = {autogen_final!r}",
)
s.check(
    "autogen_final_references_endosymbiotic",
    lambda: "endosymbiotic" in autogen_final.lower(),
    msg=f"final = {autogen_final!r}",
)
s.check(
    "crewai_reaches_same_final_answer",
    lambda: crewai_final == autogen_final,
    msg=f"crewai={crewai_final!r}  autogen={autogen_final!r}",
)
s.check(
    "autogen_terminates_when_critic_returns_empty",
    lambda: autogen_log[-1].sender == "critic" and "none" in autogen_log[-1].content,
    msg=f"last message = {autogen_log[-1]}",
)
s.check(
    "crewai_trace_includes_all_three_roles",
    lambda: all(role in str(crewai_trace) for role in ("drafter", "critic", "reviser")),
    msg=f"trace = {crewai_trace}",
)


## Exercises

1. **Cyclic task DAG.** Real CrewAI supports `manager` processes that
   can loop. Extend the Crew class with a conditional edge: "if critic
   finds issues, reschedule reviser + critic". This converges to the
   AutoGen behaviour.
2. **Multi-critic voting.** Add three critics, each focusing on a
   different dimension (clarity, accuracy, style). The reviser only
   acts on issues flagged by at least two critics.
3. **Real AutoGen.** `pip install autogen-agentchat` and run the same
   flow with `RoundRobinGroupChat`. The message routing is identical.

## References

- AutoGen 0.4 docs — the new event-driven model supersedes the older
  conversable-agent API.
- CrewAI's *Process* abstraction (sequential, hierarchical) for the
  production routing semantics.


In [ ]:
s.summary()
s.save()
